In [ ]:
import os, subprocess, sys

REPO_URL = 'https://github.com/NguyenAn05/pocket-r1.git'
REPO_DIR = '/tmp/pocket-r1'
ONPOLICY_DIR = '/kaggle/input/pocket-r1-onpolicy-v1'
PRM_DATA_DIR = '/tmp/prm_v1'
OUT_PRM_DIR = '/kaggle/working/deberta_prm_v1'

print(subprocess.check_output(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv'], text=True))
print('On-policy dataset files:', os.listdir(ONPOLICY_DIR))

if os.path.exists(REPO_DIR):
    subprocess.check_call(['rm', '-rf', REPO_DIR])
subprocess.check_call(['git', 'clone', '--depth', '1', REPO_URL, REPO_DIR])

subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR])
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'sentencepiece', 'protobuf'])
print('Phase 3 environment ready!')


In [ ]:
import os, shutil, subprocess, sys

os.makedirs(OUT_PRM_DIR, exist_ok=True)
subprocess.check_call([
    sys.executable, f'{REPO_DIR}/scripts/build_prm_dataset.py',
    '--clean-examples', '8000',
    '--synthetic-corruptions', '10000',
    '--base-seed', '950000',
    '--equity-samples', '48',
    '--onpolicy-files',
    f'{ONPOLICY_DIR}/exp0_trajectories.jsonl',
    f'{ONPOLICY_DIR}/exp1b_trajectories.jsonl',
    '--output-dir', PRM_DATA_DIR,
], cwd=REPO_DIR)

shutil.copy(f'{PRM_DATA_DIR}/stats_report.json', f'{OUT_PRM_DIR}/dataset_stats_report.json')
print('PRM v1 dataset built!')


In [ ]:
import os, subprocess, sys

train_env = os.environ.copy()
train_env['CUDA_VISIBLE_DEVICES'] = '0'
subprocess.check_call([
    sys.executable, f'{REPO_DIR}/scripts/train_deberta_prm.py',
    '--config', f'{REPO_DIR}/configs/prm/deberta_v3_base.yaml',
    '--train-file', f'{PRM_DATA_DIR}/train.jsonl',
    '--val-synthetic-file', f'{PRM_DATA_DIR}/val_synthetic.jsonl',
    '--val-onpolicy-file', f'{PRM_DATA_DIR}/val_onpolicy_real.jsonl',
    '--output-dir', OUT_PRM_DIR,
    '--epochs', '3',
    '--batch-size', '8',
], cwd=REPO_DIR, env=train_env)
print('DeBERTa-v3 PRM v1 training & evaluation complete! Files:', os.listdir(OUT_PRM_DIR))
